In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, fdf91a3a-9f60-4c6a-b99d-59fecaf094c1, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, fdf91a3a-9f60-4c6a-b99d-59fecaf094c1, 4, Finished, Available, Finished, False)

16 projects found


In [3]:
all_payment_applications = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling payment applications for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/payment_applications",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_payment_applications.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total payment applications: {len(all_payment_applications)}")

StatementMeta(, fdf91a3a-9f60-4c6a-b99d-59fecaf094c1, 5, Finished, Available, Finished, False)

Pulling payment applications for: 1100 Fulton Street
Pulling payment applications for: 11 ESSEX ST
Pulling payment applications for: 337A & 337B West Broadway Rehabilitaion Work
Pulling payment applications for: 360 Lexington 8th & 20th Floor
Pulling payment applications for: 549 Munroe Av
Pulling payment applications for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling payment applications for: Boys & Girls Club
Pulling payment applications for: EMBANKMENT PHASE II
Pulling payment applications for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling payment applications for: Lillipvt 45 Renwick St
Pulling payment applications for: PCNA 711 11TH AVE
Pulling payment applications for: Sandbox Test Project
Pulling payment applications for: Standard Project Template
Pulling payment applications for: SYMRISE - 15th & 16th Flr
Pulling payment applications for: TEST - ABM SUBORDINATE
  Error 403, skipping
Pulling payment applications for: VOCO HOTEL TSQ
Done! Total payment applica

In [4]:
import pandas as pd
import re

clean_rows = []
for row in all_payment_applications:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_payment_applications_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_payment_applications_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, fdf91a3a-9f60-4c6a-b99d-59fecaf094c1, 6, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
